# 2.3 Solving Systems of Linear Equations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muratakcs/math-for-ai/blob/main/notebooks/ch02/2.3-solving-linear-systems.ipynb)
[![Run in your browser](https://img.shields.io/badge/run_in_browser-JupyterLite-F37626?logo=jupyter)](https://muratakcs.github.io/math-for-ai/lite/lab/index.html?path=ch02/2.3-solving-linear-systems.ipynb)

*Mathematics for AI* · Based on Section 2.3 of Deisenroth, Faisal & Ong, **Mathematics for Machine Learning** (CUP 2020), [mml-book.com](https://mml-book.com)

**What you will be able to do after this notebook**

1. Describe the solution set of $A\boldsymbol{x}=\boldsymbol{b}$ as *particular solution + all solutions of $A\boldsymbol{x}=\boldsymbol{0}$*.
2. Run Gaussian elimination by hand **and** in code, step by step.
3. Read basic/free variables, a particular solution and the null space off the (reduced) row-echelon form, using the *Minus-1 trick*.
4. Compute a matrix inverse with $[A\,|\,I] \rightsquigarrow [I\,|\,A^{-1}]$.
5. Explain why computers need **pivoting**, why elimination costs $O(n^3)$, and when **iterative methods** are used instead.

> **How to use this notebook.** Whenever you see 🤔 **Predict**, stop and write down your answer *before* running the next cell. Cells marked ✍️ are exercises for you.

## 0. Setup

Run this cell first. On Colab and local Jupyter everything is already installed; in the browser version (JupyterLite) it installs the packages, which takes a few seconds the first time.

In [ ]:
# Install packages when running in the browser (JupyterLite / Pyodide); no-op elsewhere.
try:
    import piplite
    await piplite.install(["numpy", "sympy", "matplotlib", "ipywidgets"])
except ImportError:
    pass

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

sp.init_printing()
print("Ready: numpy", np.__version__, "| sympy", sp.__version__)

The next cell only contains **display helpers** (pretty-printing matrices as LaTeX and a step-through slider). They are not part of the mathematics, so feel free to skip reading them.

In [ ]:
# ---------- Display helpers (not part of the math) ----------
def latex_matrix(M, ncoef=None, highlight=None):
    """LaTeX for a matrix; a vertical bar is drawn after `ncoef` columns (augmented matrix).
    `highlight=(i, j)` draws a box around entry (i, j), e.g. the current pivot."""
    M = sp.Matrix(M)
    rows, cols = M.shape
    ncoef = cols if ncoef is None else ncoef
    spec = "r" * ncoef + ("|" + "r" * (cols - ncoef) if cols > ncoef else "")
    lines = []
    for i in range(rows):
        cells = []
        for j in range(cols):
            s = sp.latex(M[i, j])
            if highlight == (i, j):
                s = r"\boxed{" + s + "}"
            cells.append(s)
        lines.append(" & ".join(cells))
    return r"\left[\begin{array}{" + spec + "}" + r" \\ ".join(lines) + r"\end{array}\right]"

def show(M, ncoef=None, label="", highlight=None):
    """Display a (possibly augmented) matrix with an optional LaTeX label in front."""
    display(Math(label + (r"\qquad " if label else "") + latex_matrix(M, ncoef, highlight)))

def show_steps(steps, ncoef=None):
    """Print every recorded step of an elimination."""
    for k, (desc, M, piv) in enumerate(steps):
        show(M, ncoef, label=rf"\text{{step {k}: }}\ {desc}", highlight=piv)

def step_through(steps, ncoef=None):
    """Interactive version of show_steps: a slider plus previous/next buttons.
    Works when the notebook is running (Colab, JupyterLite, local Jupyter)."""
    import ipywidgets as w
    out = w.Output()
    slider = w.IntSlider(min=0, max=len(steps) - 1, value=0, description="step")
    prev_btn, next_btn = w.Button(description="◀ previous"), w.Button(description="next ▶")
    def render(_=None):
        desc, M, piv = steps[slider.value]
        with out:
            out.clear_output(wait=True)
            show(M, ncoef, label=rf"\text{{step {slider.value}: }}\ {desc}", highlight=piv)
    prev_btn.on_click(lambda _: setattr(slider, "value", max(slider.value - 1, slider.min)))
    next_btn.on_click(lambda _: setattr(slider, "value", min(slider.value + 1, slider.max)))
    slider.observe(render, names="value")
    render()
    display(w.VBox([w.HBox([prev_btn, slider, next_btn]), out]))

def check(name, condition):
    """Tiny test helper: prints a green tick or a red cross instead of raising an error."""
    print(("✅ " if condition else "❌ ") + name)

## 1. Particular and general solution (§2.3.1)

Consider the system with 2 equations and 4 unknowns

$$
\begin{bmatrix}1&0&8&-4\\0&1&2&12\end{bmatrix}
\begin{bmatrix}x_1\\x_2\\x_3\\x_4\end{bmatrix}
=\begin{bmatrix}42\\8\end{bmatrix}.
$$

Recall the **column view**: $A\boldsymbol{x} = \sum_i x_i\boldsymbol{c}_i$ is a linear combination of the columns $\boldsymbol{c}_i$ of $A$. Because the first two columns are the unit vectors, we can read off a **particular solution**: $\boldsymbol{b} = 42\,\boldsymbol{c}_1 + 8\,\boldsymbol{c}_2$, i.e. $\boldsymbol{x}_p = [42, 8, 0, 0]^\top$.

We use `sympy` so that all arithmetic is **exact** (fractions instead of floating point numbers).

In [ ]:
A = sp.Matrix([[1, 0, 8, -4],
               [0, 1, 2, 12]])
b = sp.Matrix([42, 8])

x_p = sp.Matrix([42, 8, 0, 0])
show(A * x_p, label=r"A\,\boldsymbol{x}_p =")
check("x_p solves Ax = b", A * x_p == b)

To find *all* solutions we look for non-trivial ways to write $\boldsymbol{0}$ as a combination of the columns. Column 3 is $8\boldsymbol{c}_1 + 2\boldsymbol{c}_2$, so $8\boldsymbol{c}_1 + 2\boldsymbol{c}_2 - \boldsymbol{c}_3 = \boldsymbol{0}$; similarly $-4\boldsymbol{c}_1 + 12\boldsymbol{c}_2 - \boldsymbol{c}_4 = \boldsymbol{0}$.

🤔 **Predict:** if we add $\lambda_1[8,2,-1,0]^\top + \lambda_2[-4,12,0,-1]^\top$ to $\boldsymbol{x}_p$, for which values of $\lambda_1, \lambda_2$ is the result still a solution?

In [ ]:
h1 = sp.Matrix([8, 2, -1, 0])
h2 = sp.Matrix([-4, 12, 0, -1])
check("A h1 = 0", A * h1 == sp.zeros(2, 1))
check("A h2 = 0", A * h2 == sp.zeros(2, 1))

# Try 5 random choices of (lambda1, lambda2): every one gives a solution.
rng = np.random.default_rng(0)
for _ in range(5):
    l1, l2 = (sp.Rational(int(v), 7) for v in rng.integers(-20, 20, size=2))
    x = x_p + l1 * h1 + l2 * h2
    print(f"lambda = ({str(l1):>6}, {str(l2):>6})  ->  A x = {list(A * x)}")

# And symbolically, for ALL lambda at once:
l1, l2 = sp.symbols("lambda_1 lambda_2")
x_general = x_p + l1 * h1 + l2 * h2
show(sp.simplify(A * x_general), label=r"A\,(\boldsymbol{x}_p+\lambda_1\boldsymbol{h}_1+\lambda_2\boldsymbol{h}_2)=")

**General recipe** (works for every linear system):

1. Find a **particular solution** of $A\boldsymbol{x}=\boldsymbol{b}$.
2. Find **all solutions** of the homogeneous system $A\boldsymbol{x}=\boldsymbol{0}$.
3. **Combine** them: $\{\boldsymbol{x}_p + \boldsymbol{h} : A\boldsymbol{h} = \boldsymbol{0}\}$.

This example was easy because of the special shape of $A$. **Gaussian elimination** brings *any* system into such a shape.

## 2. Elementary transformations (§2.3.2)

Three operations change the system but **not its solution set**:

1. exchange two rows,
2. multiply a row by a constant $\lambda \neq 0$,
3. add a multiple of one row to another row.

We work on the **augmented matrix** $[A\,|\,\boldsymbol{b}]$. Each function below returns the new matrix *and* a LaTeX description of what it did, so that we can record and replay every step later. Rows are numbered from 1 in the descriptions (as in the book) but from 0 in Python.

In [ ]:
def swap(M, i, j):
    "Exchange rows i and j."
    M = M.copy()
    M.row_swap(i, j)
    return M, rf"R_{i+1} \leftrightarrow R_{j+1}"

def scale(M, i, c):
    "Multiply row i by the nonzero constant c."
    assert c != 0, "scaling by 0 is not allowed: it would lose information"
    M = M.copy()
    M[i, :] = (c * M[i, :]).applyfunc(sp.expand)
    return M, rf"R_{i+1} \leftarrow {sp.latex(c)}\,R_{i+1}"

def add_multiple(M, src, dst, c):
    "Add c times row src to row dst."
    M = M.copy()
    M[dst, :] = (M[dst, :] + c * M[src, :]).applyfunc(sp.expand)
    sign = "+" if c >= 0 else "-"
    return M, rf"R_{dst+1} \leftarrow R_{dst+1} {sign} {sp.latex(abs(c))}\,R_{src+1}"

🤔 **Predict:** $\boldsymbol{x} = [1,1,1]^\top$ solves the system below. Will it still solve the system after a random sequence of elementary operations?

In [ ]:
Ab = sp.Matrix([[1,  1, 1, 3],
                [1, -1, 2, 2],
                [0,  1, 1, 2]])
x_known = sp.Matrix([1, 1, 1])

M = Ab
show(M, ncoef=3, label=r"\text{start}")
for op in [lambda M: swap(M, 0, 2),
           lambda M: scale(M, 1, sp.Rational(-1, 3)),
           lambda M: add_multiple(M, 0, 2, -5)]:
    M, desc = op(M)
    show(M, ncoef=3, label=desc)
    residual = M[:, :3] * x_known - M[:, 3]
    check("x = [1,1,1] still solves the system", residual == sp.zeros(3, 1))

## 3. Row-echelon form

A matrix is in **row-echelon form (REF)** if

* all rows containing only zeros are at the bottom, and
* the first nonzero entry of each nonzero row (the **pivot**) is strictly to the right of the pivot of the row above.

This gives the typical "staircase" shape. Turning the definition into code is a good way to make sure we understand it:

In [ ]:
def first_nonzero(row):
    "Index of the first nonzero entry of a row, or None for a zero row."
    return next((j for j, v in enumerate(row) if v != 0), None)

def is_row_echelon(M):
    M = sp.Matrix(M)
    last_pivot, seen_zero_row = -1, False
    for i in range(M.rows):
        p = first_nonzero(list(M.row(i)))
        if p is None:
            seen_zero_row = True          # from now on only zero rows are allowed
            continue
        if seen_zero_row or p <= last_pivot:
            return False
        last_pivot = p
    return True

check("staircase is REF",      is_row_echelon([[1, 2, 3], [0, 0, 4], [0, 0, 0]]) is True)
check("pivot below pivot",     is_row_echelon([[1, 2, 3], [0, 4, 5], [0, 6, 7]]) is False)
check("zero row in the middle", is_row_echelon([[1, 2, 3], [0, 0, 0], [0, 0, 4]]) is False)

## 4. Gaussian elimination, step by step

**Algorithm** for an augmented matrix $[A\,|\,\boldsymbol{b}]$ with `ncoef` coefficient columns:

1. Go through the columns from left to right. In the current column, find a row (at or below the current row) with a nonzero entry. If there is none, the column has no pivot: move on.
2. Swap that row up, so that it becomes the **pivot row**.
3. Use operation 3 to create **zeros below the pivot**.
4. Continue with the next row and next column.
5. *(Reduced form only)* Scale every pivot to 1 and create zeros **above** each pivot, working from the last pivot upwards.

The function records every intermediate matrix in `steps`, so that we can replay the algorithm.

In [ ]:
def gaussian_elimination(Ab, ncoef, reduced=False):
    """Bring the (augmented) matrix Ab into REF (or RREF if reduced=True).
    Returns: the final matrix, the list of pivot positions (row, col), and the recorded steps."""
    M = sp.Matrix(Ab)
    steps = [(r"\text{start}", M, None)]
    pivots, r = [], 0                        # r = row where the next pivot goes
    for c in range(ncoef):
        # 1. find a pivot in column c, at or below row r
        p = next((i for i in range(r, M.rows) if M[i, c] != 0), None)
        if p is None:
            continue                         # no pivot in this column -> free variable
        # 2. move it up
        if p != r:
            M, desc = swap(M, r, p)
            steps.append((desc, M, (r, c)))
        # 3. eliminate everything below the pivot
        for i in range(r + 1, M.rows):
            if M[i, c] != 0:
                M, desc = add_multiple(M, r, i, -M[i, c] / M[r, c])
                steps.append((desc, M, (r, c)))
        pivots.append((r, c))
        r += 1
        if r == M.rows:
            break
    if reduced:
        # 5. back to front: pivots become 1, entries above pivots become 0
        for (r_, c) in reversed(pivots):
            if M[r_, c] != 1:
                M, desc = scale(M, r_, 1 / M[r_, c])
                steps.append((desc, M, (r_, c)))
            for i in range(r_):
                if M[i, c] != 0:
                    M, desc = add_multiple(M, r_, i, -M[i, c])
                    steps.append((desc, M, (r_, c)))
    return M, pivots, steps

### Example 2.6 from the book

$$
\begin{aligned}
-2x_1+4x_2-2x_3-x_4+4x_5&=-3\\
4x_1-8x_2+3x_3-3x_4+x_5&=2\\
x_1-2x_2+x_3-x_4+x_5&=0\\
x_1-2x_2-3x_4+4x_5&=a
\end{aligned}
$$

Note that the right-hand side contains a **parameter** $a$. Because `sympy` computes symbolically, we can carry $a$ through the whole elimination.

🤔 **Predict:** the system has 4 equations and 5 unknowns. Can it have a unique solution?

In [ ]:
a = sp.symbols("a")
Ab = sp.Matrix([[-2,  4, -2, -1, 4, -3],
                [ 4, -8,  3, -3, 1,  2],
                [ 1, -2,  1, -1, 1,  0],
                [ 1, -2,  0, -3, 4,  a]])

REF, pivots, steps = gaussian_elimination(Ab, ncoef=5)
show_steps(steps, ncoef=5)

The boxed entry is the pivot that the current step uses. **Try the interactive version** (runs only in a live notebook: Colab, JupyterLite or local Jupyter):

In [ ]:
step_through(steps, ncoef=5)

Our algorithm picks the *first* available row as pivot row, whereas the book first swaps rows 1 and 3 for convenience. The intermediate matrices therefore differ, but the conclusion is the same. Look at the **last row**:

In [ ]:
show(REF.row(-1), ncoef=5, label=r"\text{last row:}")
print("The last row reads  0 = ", REF[-1, -1])
print("Solvable only if", sp.solve(sp.Eq(REF[-1, -1], 0), a))
print("Pivot positions (row, col), 0-based:", pivots)

So the system is solvable **only for $a=-1$**. The pivot columns are columns 1, 3 and 4 (1-based): $x_1, x_3, x_4$ are **basic variables**, while $x_2, x_5$ are **free variables**. With 2 free variables, a solvable system has infinitely many solutions; the answer to the prediction is *no*.

## 5. Reduced row-echelon form and the Minus-1 trick

A matrix is in **reduced row-echelon form (RREF)** if it is in REF, every pivot equals 1, and each pivot is the only nonzero entry in its column.

From the RREF everything can be *read off*:

* **Particular solution:** set all free variables to 0; each basic variable equals the right-hand side entry of its pivot row.
* **Null space** (all solutions of $A\boldsymbol{x}=\boldsymbol{0}$) with the **Minus-1 trick**: extend the $k\times n$ RREF (without zero rows) to an $n \times n$ matrix $\tilde A$ by inserting rows $[0 \cdots 0\; -1\; 0 \cdots 0]$ so that the diagonal contains only $1$ or $-1$. The columns with $-1$ on the diagonal are a basis of the null space.

In [ ]:
def minus_one_trick(R, pivots):
    "R: RREF of the coefficient matrix. Returns the extended matrix and the null-space basis."
    n = R.cols
    pivot_row_of = {c: r for r, c in pivots}       # pivot column -> its row
    A_tilde = sp.zeros(n, n)
    for j in range(n):
        if j in pivot_row_of:
            A_tilde[j, :] = R[pivot_row_of[j], :]  # copy the pivot row
        else:
            A_tilde[j, j] = -1                     # insert a "-1 row"
    basis = [A_tilde[:, j] for j in range(n) if j not in pivot_row_of]
    return A_tilde, basis

**Example 2.7 / 2.8** in the book: $A$ is already in RREF.

In [ ]:
R = sp.Matrix([[1, 3, 0, 0,  3],
               [0, 0, 1, 0,  9],
               [0, 0, 0, 1, -4]])
_, piv, _ = gaussian_elimination(R, ncoef=5, reduced=True)
A_tilde, basis = minus_one_trick(R, piv)
show(A_tilde, label=r"\tilde{A} =")
for v in basis:
    show(v.T, label=r"\text{null-space vector}^\top:")
    check("R v = 0", R * v == sp.zeros(3, 1))

Now we put everything together into a function that returns the **general solution** of any system, or reports that there is none.

In [ ]:
def solve_general(A, b):
    """General solution of A x = b: returns (x_p, [h_1, ..., h_k]) with
    solution set { x_p + sum_i lambda_i h_i }, or (None, []) if the system is inconsistent."""
    A, b = sp.Matrix(A), sp.Matrix(b)
    n = A.cols
    R, pivots, _ = gaussian_elimination(A.row_join(b), ncoef=n, reduced=True)
    # A row [0 ... 0 | c] with c != 0 means 0 = c: no solution.
    for i in range(R.rows):
        if all(R[i, j] == 0 for j in range(n)) and R[i, n] != 0:
            return None, []
    x_p = sp.zeros(n, 1)
    for r, c in pivots:
        x_p[c] = R[r, n]                              # free variables stay 0
    _, null_basis = minus_one_trick(R[:len(pivots), :n], pivots)
    return x_p, null_basis

def show_general(x_p, basis):
    if x_p is None:
        display(Markdown("**No solution** (inconsistent system)."))
        return
    terms = latex_matrix(x_p) + "".join(
        rf" + \lambda_{k+1}" + latex_matrix(h) for k, h in enumerate(basis))
    display(Math(r"\boldsymbol{x} = " + terms))

Back to Example 2.6 with $a=-1$. The book obtains the particular solution $[2,0,-1,1,0]^\top$ and the null-space vectors $[2,1,0,0,0]^\top$ and $[2,0,-1,2,1]^\top$. Do we?

In [ ]:
A26 = Ab[:, :5]
b26 = Ab[:, 5].subs(a, -1)
x_p, basis = solve_general(A26, b26)
show_general(x_p, basis)

check("A x_p = b", A26 * x_p == b26)
for h in basis:
    check("A h = 0", A26 * h == sp.zeros(4, 1))

🤔 Our null-space vectors are the **negatives** of the book's. Is that a problem?

No: any basis of the null space describes the same set of solutions, because $\lambda$ ranges over all of $\mathbb{R}$ (just replace $\lambda$ by $-\lambda$). We can check that both pairs of vectors span the same space: stacking all four vectors as columns still gives rank 2.

In [ ]:
book_basis = [sp.Matrix([2, 1, 0, 0, 0]), sp.Matrix([2, 0, -1, 2, 1])]
check("same particular solution as the book", x_p == sp.Matrix([2, 0, -1, 1, 0]))
check("book's vectors and ours span the same space",
      sp.Matrix.hstack(*basis, *book_basis).rank() == 2)

### No, one, or infinitely many solutions

🤔 **Predict** for each system of Example 2.2 (book §2.1) whether it has no, exactly one, or infinitely many solutions — then run the cell.

| | (a) | (b) | (c) |
|---|---|---|---|
| eq. 1 | $x_1+x_2+x_3=3$ | $x_1+x_2+x_3=3$ | $x_1+x_2+x_3=3$ |
| eq. 2 | $x_1-x_2+2x_3=2$ | $x_1-x_2+2x_3=2$ | $x_1-x_2+2x_3=2$ |
| eq. 3 | $2x_1+3x_3=1$ | $x_2+x_3=2$ | $2x_1+3x_3=5$ |

In [ ]:
systems = {
    "(a)": ([[1, 1, 1], [1, -1, 2], [2, 0, 3]], [3, 2, 1]),
    "(b)": ([[1, 1, 1], [1, -1, 2], [0, 1, 1]], [3, 2, 2]),
    "(c)": ([[1, 1, 1], [1, -1, 2], [2, 0, 3]], [3, 2, 5]),
}
for name, (A_, b_) in systems.items():
    display(Markdown(f"**System {name}**"))
    show_general(*solve_general(A_, b_))

## 6. Calculating the inverse

$AA^{-1}=I$ means solving $n$ systems at once, one for each column of $I$. Hence

$$[A\,|\,I] \;\rightsquigarrow\; [I\,|\,A^{-1}].$$

If the left block cannot be turned into $I$, then $A$ is not invertible.

In [ ]:
def inverse_gauss(A):
    A = sp.Matrix(A)
    n = A.rows
    R, pivots, steps = gaussian_elimination(A.row_join(sp.eye(n)), ncoef=n, reduced=True)
    if R[:, :n] != sp.eye(n):
        return None, steps                    # not invertible
    return R[:, n:], steps

A29 = sp.Matrix([[1, 0, 2, 0],
                 [1, 1, 0, 0],
                 [1, 2, 0, 1],
                 [1, 1, 1, 1]])               # Example 2.9
A29_inv, steps = inverse_gauss(A29)
print(f"{len(steps) - 1} elementary operations; first and last step:")
show(steps[0][1], ncoef=4, label=r"[A\,|\,I] =")
show(steps[-1][1], ncoef=4, label=r"[I\,|\,A^{-1}] =")
check("A @ A_inv == I", A29 * A29_inv == sp.eye(4))
check("matches numpy", np.allclose(np.linalg.inv(np.array(A29, dtype=float)),
                                   np.array(A29_inv, dtype=float)))

In [ ]:
step_through(steps, ncoef=4)     # replay the inversion step by step

🤔 **Predict:** what happens for the singular matrix $\begin{bmatrix}1&2\\2&4\end{bmatrix}$?

In [ ]:
inv, steps = inverse_gauss([[1, 2], [2, 4]])
print("inverse:", inv)
show(steps[-1][1], ncoef=2, label=r"\text{final:}")

## 7. What changes on a real computer: floating point and pivoting

So far we computed **exactly**. Real ML code uses floating-point numbers (`float64`), and then *which* pivot we choose suddenly matters. Below is a plain `numpy` implementation, with an option for **partial pivoting**: always choose the entry with the **largest absolute value** in the column as pivot.

In [ ]:
def solve_float(A, b, pivoting=True):
    "Gaussian elimination + back substitution in floating point (square, invertible A)."
    M = np.column_stack([A, b]).astype(float)
    n = len(b)
    for k in range(n):
        if pivoting:
            p = k + np.argmax(np.abs(M[k:, k]))   # largest entry in column k
            M[[k, p]] = M[[p, k]]
        for i in range(k + 1, n):
            M[i, k:] -= (M[i, k] / M[k, k]) * M[k, k:]
    x = np.zeros(n)                                # back substitution
    for i in reversed(range(n)):
        x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:]) / M[i, i]
    return x

🤔 **Predict:** the exact solution of
$\;10^{-20}x_1 + x_2 = 1,\;\; x_1 + x_2 = 2\;$
is very close to $x_1 = x_2 = 1$. What will plain elimination *without* pivoting return?

In [ ]:
A_bad = np.array([[1e-20, 1.0],
                  [1.0,   1.0]])
b_bad = np.array([1.0, 2.0])

print("exact (sympy):     ", [float(v) for v in sp.Matrix([[sp.Rational(1, 10**20), 1], [1, 1]]).solve(sp.Matrix([1, 2]))])
print("no pivoting:       ", solve_float(A_bad, b_bad, pivoting=False))
print("partial pivoting:  ", solve_float(A_bad, b_bad, pivoting=True))
print("np.linalg.solve:   ", np.linalg.solve(A_bad, b_bad))

Without pivoting we divide by the tiny pivot $10^{-20}$, which creates a multiplier of $10^{20}$. In the second row $1 - 10^{20}$ is rounded to $-10^{20}$: the information in the original entries is **lost**, and $x_1$ comes out as $0$. Pivoting avoids huge multipliers. Libraries such as `np.linalg.solve` (LAPACK) always pivot.

> **Lesson:** two algorithms that are identical in exact arithmetic can behave very differently on a computer. This is why *numerical* linear algebra is its own field.

## 8. How expensive is Gaussian elimination?

For an $n\times n$ system, eliminating column $k$ touches roughly $(n-k)^2$ entries. Summing over $k$ gives about $n^3/3$ multiply–add operations, i.e. **$O(n^3)$**. Let us count the operations exactly instead of trusting the formula:

In [ ]:
def count_operations(n):
    "Number of multiply-add operations in forward elimination of an n x n system (with right-hand side)."
    ops = 0
    for k in range(n):
        for i in range(k + 1, n):
            ops += (n + 1 - k)            # update of row i, columns k..n
    return ops

ns = np.array([10, 20, 50, 100, 200, 500, 1000])
counts = np.array([count_operations(n) for n in ns])

plt.figure(figsize=(6, 4))
plt.loglog(ns, counts, "o-", label="counted operations")
plt.loglog(ns, ns**3 / 3, "--", label=r"$n^3/3$")
plt.xlabel("n (number of unknowns)"); plt.ylabel("operations")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.title("Gaussian elimination is cubic")
plt.show()

for n, c in zip(ns, counts):
    print(f"n = {n:5d}:  {c:>14,d} operations   (ratio to n^3/3: {c / (n**3 / 3):.3f})")

🤔 **Predict:** if solving a system with $n=1000$ takes 0.05 s, roughly how long does $n=10\,000$ take? And $n = 10^6$ (a modest number of parameters in ML)?

*(Answer: $\times 1000$ and $\times 10^9$: about 50 s and over a year. For such sizes we need other methods.)*

## 9. Iterative methods

For very large systems we do not solve exactly but **iterate**:
$\boldsymbol{x}^{(k+1)} = C\boldsymbol{x}^{(k)} + \boldsymbol{d}$, which converges to the solution $\boldsymbol{x}_*$ when $\|C\| < 1$, because
$\|\boldsymbol{x}^{(k+1)}-\boldsymbol{x}_*\| \le \|C\|\,\|\boldsymbol{x}^{(k)}-\boldsymbol{x}_*\|$.

The simplest example is the **Jacobi method**. Split $A = D + R$, with $D$ the diagonal and $R$ the rest. Then $A\boldsymbol{x} = \boldsymbol{b}$ becomes $\boldsymbol{x} = D^{-1}(\boldsymbol{b} - R\boldsymbol{x})$, i.e. $C = -D^{-1}R$ and $\boldsymbol{d} = D^{-1}\boldsymbol{b}$. Each step only needs a matrix–vector product, $O(n^2)$, or even less for sparse matrices.

In [ ]:
A_j = np.array([[4., 1, 1],
                [1, 5, 2],
                [1, 2, 6]])                  # diagonally dominant
b_j = np.array([6., 8, 9])
x_star = np.linalg.solve(A_j, b_j)

D = np.diag(np.diag(A_j))
C = -np.linalg.inv(D) @ (A_j - D)
d = np.linalg.inv(D) @ b_j
print("||C||_inf =", np.abs(C).sum(axis=1).max(), " (< 1, so Jacobi converges)")

x = np.zeros(3)
errors = []
for k in range(25):
    x = C @ x + d
    errors.append(np.linalg.norm(x - x_star))

plt.figure(figsize=(6, 3.5))
plt.semilogy(errors, "o-")
plt.xlabel("iteration k"); plt.ylabel(r"$\|x^{(k)} - x_*\|$")
plt.title("Jacobi iteration: error shrinks geometrically"); plt.grid(alpha=0.3)
plt.show()
print("after 25 iterations:", x, "  exact:", x_star)

In ML, *gradient descent* plays the same role for problems that are too big (or not linear at all): many cheap iterations instead of one expensive exact solve.

## 10. ✍️ Exercises

Each exercise has a test cell that prints ✅ or ❌. Solutions are hidden below each exercise; try first!

### Exercise 1: back substitution

Complete `back_substitution(U, c)`, which solves $U\boldsymbol{x} = \boldsymbol{c}$ for an **upper triangular** $U$ with nonzero diagonal entries. Work from the last row upwards. *(Hint: look at the end of `solve_float`.)*

In [ ]:
def back_substitution(U, c):
    U, c = np.asarray(U, float), np.asarray(c, float)
    n = len(c)
    x = np.zeros(n)
    # YOUR CODE HERE
    return None

In [ ]:
U_test = np.array([[2., 1, -1], [0, 3, 2], [0, 0, 4]])
c_test = np.array([1., 13, 8])
res = back_substitution(U_test, c_test)
if res is None:
    print("⏳ not implemented yet")
else:
    check("solves U x = c", np.allclose(U_test @ res, c_test))

<details>
<summary><b>Show solution</b></summary>

```python
def back_substitution(U, c):
    U, c = np.asarray(U, float), np.asarray(c, float)
    n = len(c)
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (c[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x
```
The solution of the test system is $\boldsymbol{x} = [0, 3, 2]^\top$.
</details>

### Exercise 2: rank via Gaussian elimination

The **rank** of a matrix is the number of pivots in its row-echelon form (§2.6). Complete `my_rank` using `gaussian_elimination`, and compare with Example 2.18 of the book.

In [ ]:
def my_rank(A):
    A = sp.Matrix(A)
    # YOUR CODE HERE
    return None

In [ ]:
tests = [([[1, 0, 1], [0, 1, 1], [0, 0, 0]], 2),
         ([[1, 2, 1], [-2, -3, 1], [3, 5, 0]], 2),
         ([[1, 2], [2, 4], [3, 6]], 1),
         (np.eye(4).astype(int).tolist(), 4)]
if my_rank(tests[0][0]) is None:
    print("⏳ not implemented yet")
else:
    for M_, expected in tests:
        check(f"rank = {expected}", my_rank(M_) == expected == np.linalg.matrix_rank(np.array(M_, float)))

<details>
<summary><b>Show solution</b></summary>

```python
def my_rank(A):
    A = sp.Matrix(A)
    _, pivots, _ = gaussian_elimination(A, ncoef=A.cols)
    return len(pivots)
```
</details>

### Exercise 3: a parameter

🤔 **Predict first:** for which $c$ is the system $\;x_1 + x_2 = 1,\; 2x_1 + 2x_2 = c\;$ solvable? How many solutions does it have then?

Then use `gaussian_elimination` with a symbolic `c = sp.symbols("c")` and read the answer off the last row, as we did for Example 2.6.

In [ ]:
c = sp.symbols("c")
# YOUR CODE HERE

<details>
<summary><b>Show solution</b></summary>

```python
R, piv, steps = gaussian_elimination(sp.Matrix([[1, 1, 1], [2, 2, c]]), ncoef=2)
show_steps(steps, ncoef=2)          # last row: [0 0 | c - 2]
```
The system is solvable only for $c = 2$. Then $x_2$ is free, so there are infinitely many solutions $\boldsymbol{x} = [1,0]^\top + \lambda[-1,1]^\top$.
</details>

### Exercise 4: break it

Find a $3\times 3$ system for which `solve_float(..., pivoting=False)` **crashes** (division by zero), although the matrix is invertible. Does pivoting fix it?

<details>
<summary><b>Show a hint</b></summary>

Put a $0$ in the top-left corner, e.g. $A = \begin{bmatrix}0&1&1\\1&0&1\\1&1&0\end{bmatrix}$. Without pivoting the first pivot is $0$; with pivoting a row swap fixes it.
</details>

## Summary

* Solution set of $A\boldsymbol{x}=\boldsymbol{b}$ = **particular solution + null space of $A$** (or empty).
* **Elementary row operations** do not change the solution set; **Gaussian elimination** uses them to reach (reduced) row-echelon form.
* From the RREF: **pivot columns ↔ basic variables**, other columns ↔ free variables; particular solution by setting free variables to 0; null space with the **Minus-1 trick**.
* Inverse: $[A\,|\,I] \rightsquigarrow [I\,|\,A^{-1}]$; it exists iff every column has a pivot.
* On a computer: always **pivot**; elimination costs $O(n^3)$; for huge systems use **iterative methods**. In practice, call `np.linalg.solve` (and almost never compute $A^{-1}$ explicitly).

**Next notebook:** 2.4 Vector spaces.